# Ch 27 · Rays, Lidar and Batch Rendering for Thousands of Worlds

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/27-rays-lidar-and-batch-rendering.ipynb)

Companion notebook for the chapter page `chapters/27-rays-lidar-and-batch-rendering.html` ([page](../chapters/27-rays-lidar-and-batch-rendering.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Rendering notebooks only: let glvnd find the NVIDIA EGL driver (from the official
# mujoco_warp tutorial). Must run before mujoco is imported by the setup cell below.
import os
os.environ.setdefault("MUJOCO_GL", "egl")
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
try:
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version": "1.0.0", '
                    '"ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
    print("EGL ICD file:", NVIDIA_ICD_CONFIG_PATH, "| MUJOCO_GL =", os.environ["MUJOCO_GL"])
except OSError as e:
    print("could not write the EGL ICD file (fine outside Colab):", e)

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
os.environ.setdefault("MUJOCO_GL", "egl")
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

This notebook runs every example of the chapter in order. Each example cell is the exact script from `examples/27-rays-lidar-and-batch-rendering/`. On a GPU runtime the budgets grow automatically (`ON_GPU`); timings are your own GPU numbers. The batch renderer needs no OpenGL/EGL. The EGL cell and `MUJOCO_GL=egl` at the top exist only for the optional `mujoco.Renderer` comparison cell after Example 27.5; it prints a message and moves on where EGL is missing.

## 1. One ray per world: `mjw.ray`
A ray from (2, 0, 2) straight down hits the top of a cylinder at dist 1.4; with `bodyexclude` set to the cylinder's body it hits the floor at 2.0. `dist` is the ray parameter t, in units of |vec|: pass unit directions to get metres. `-1` means no hit.

In [ ]:
# Example 27.1 - ex1_single_ray.py
"""Example 27.1 - one ray per world with mjw.ray, and bodyexclude.

A ray from (2, 0, 2) straight down hits the top of a cylinder (top at z = 0.6), so dist = 1.4.
With bodyexclude = the cylinder's body the same ray passes through it and hits the floor
(dist = 2.0). A second call gives every world its own ray origin (pnt shape (nworld, 1)).
Run: python ex1_single_ray.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

RAY_SCENE = """
<mujoco model="ray_scene">
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="post" pos="2 0 0.3">
      <geom name="post" type="cylinder" size=".2 .3" rgba=".3 .6 .9 1"/>
    </body>
    <body name="ball" pos="0 0 0.5"><freejoint/>
      <geom name="ball" type="sphere" size=".25" rgba=".9 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4

mjm = mujoco.MjModel.from_xml_string(RAY_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
mjw.kinematics(m, d)          # rays read d.geom_xpos / d.geom_xmat: compute poses first

# 1. the same ray for every world: shape (1, 1) is broadcast over worlds (worldid % 1)
pnt = wp.array([[wp.vec3(2.0, 0.0, 2.0)]], dtype=wp.vec3)
vec = wp.array([[wp.vec3(0.0, 0.0, -1.0)]], dtype=wp.vec3)   # unit length: dist is in metres
dist, geomid, normal = mjw.ray(m, d, pnt, vec)
gname = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, int(geomid.numpy()[0, 0]))
print("output shapes:", dist.shape, geomid.shape, normal.shape)
print(f"hit {gname!r}: dist = {dist.numpy()[0, 0]:.4f}, normal = {normal.numpy()[0, 0]}")

post = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "post")
dist, geomid, normal = mjw.ray(m, d, pnt, vec, bodyexclude=post)
gname = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, int(geomid.numpy()[0, 0]))
print(f"bodyexclude=post (body {post}): hit {gname!r}, dist = {dist.numpy()[0, 0]:.4f}")

# 2. one origin per world: shape (nworld, 1); world k starts at x = 2 - (2/3)(k % 4)
x = 2.0 - (2.0 / 3.0) * (np.arange(NWORLD) % 4)
pnt_w = wp.array(np.stack([x, np.zeros(NWORLD), np.full(NWORLD, 2.0)], 1)[:, None, :],
                 dtype=wp.vec3)
vec_w = wp.array(np.tile([0.0, 0.0, -1.0], (NWORLD, 1, 1)), dtype=wp.vec3)
dist, geomid, _ = mjw.ray(m, d, pnt_w, vec_w)
for w in range(min(NWORLD, 4)):
    g = int(geomid.numpy()[w, 0])
    name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g) if g >= 0 else "miss"
    print(f"world {w}: origin x = {x[w]:+.1f} -> {name:5s} dist = {dist.numpy()[w, 0]:.4f}")

# 3. a ray that misses everything: dist = -1, geomid = -1
dist, geomid, _ = mjw.ray(m, d, pnt, wp.array([[wp.vec3(0.0, 0.0, 1.0)]], dtype=wp.vec3))
print("upward ray:", "dist =", dist.numpy()[0, 0], "geomid =", geomid.numpy()[0, 0])

## 2. A 360-ray lidar in every world: `mjw.rays`
A small kernel with launch shape `(nworld, nray)` builds the rays from the robot body's `d.xpos`/`d.xmat`; one `rays` call casts them all. Without `bodyexclude` every ray starts inside the robot and reports 0.15 m. `vec6` is a private type from `mujoco_warp._src.types`.

In [ ]:
# Example 27.2 - ex2_lidar.py
"""Example 27.2 - a 360-ray horizontal lidar in every world with mjw.rays.

Each world puts the robot at its own (x, y, yaw). A small Warp kernel turns d.xpos / d.xmat of
the robot body into per-world ray origins and directions (no host round trip), then one
mjw.rays call casts nworld x 360 rays. bodyexclude keeps the lidar from seeing its own body.
Run: python ex2_lidar.py [--dump-json PATH]
  --dump-json PATH  writes the ranges and the room layout for the page's polar plots (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec6     # private API: the geomgroup mask type

np.set_printoptions(precision=3, suppress=True)

ROOM = """
<mujoco model="lidar_room">
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="3 3 .1"/>
    <geom name="wall_e" type="box" pos=" 2 0 .5" size=".05 2 .5"/>
    <geom name="wall_w" type="box" pos="-2 0 .5" size=".05 2 .5"/>
    <geom name="wall_n" type="box" pos="0  2 .5" size="2 .05 .5"/>
    <geom name="wall_s" type="box" pos="0 -2 .5" size="2 .05 .5"/>
    <geom name="pillar_a" type="cylinder" pos=" 1 1 .4" size=".2 .4"/>
    <geom name="pillar_b" type="cylinder" pos="-1 .5 .4" size=".15 .4"/>
    <geom name="crate" type="box" pos="0 -1 .2" size=".3 .2 .2" euler="0 0 30"/>
    <body name="robot" pos="0 0 .15"><freejoint/>
      <geom name="robot" type="cylinder" size=".15 .15" rgba=".2 .7 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NRAY = 360


@wp.kernel
def lidar_rays(xpos: wp.array2d(dtype=wp.vec3), xmat: wp.array2d(dtype=wp.mat33), body: int,
               local_dir: wp.array(dtype=wp.vec3),
               pnt_out: wp.array2d(dtype=wp.vec3), vec_out: wp.array2d(dtype=wp.vec3)):
    worldid, rayid = wp.tid()                       # launch shape (nworld, nray)
    pnt_out[worldid, rayid] = xpos[worldid, body]   # lidar at the body origin
    vec_out[worldid, rayid] = xmat[worldid, body] @ local_dir[rayid]   # body frame -> world


mjm = mujoco.MjModel.from_xml_string(ROOM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
robot = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "robot")

# per-world pose: (x, y, yaw); repeat a pattern of 4 so every GPU world is valid too
poses = np.array([[0.0, 0.0, 0.0], [1.2, -1.2, 45.0], [-1.4, 1.3, 90.0], [0.6, 0.9, 180.0]])
pose = poses[np.arange(NWORLD) % 4]
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 0:2] = pose[:, :2]
qpos[:, 3] = np.cos(np.radians(pose[:, 2]) / 2)   # quaternion (w, x, y, z) about z
qpos[:, 6] = np.sin(np.radians(pose[:, 2]) / 2)
d.qpos.assign(qpos.astype(np.float32))
mjw.kinematics(m, d)

ang = np.radians(np.arange(NRAY))                   # 1 degree per ray, in the body frame
local_dir = wp.array(np.stack([np.cos(ang), np.sin(ang), np.zeros(NRAY)], 1), dtype=wp.vec3)
pnt = wp.zeros((NWORLD, NRAY), dtype=wp.vec3)
vec = wp.zeros((NWORLD, NRAY), dtype=wp.vec3)
wp.launch(lidar_rays, dim=(NWORLD, NRAY), inputs=[d.xpos, d.xmat, robot, local_dir],
          outputs=[pnt, vec])

dist = wp.zeros((NWORLD, NRAY), dtype=float)
geomid = wp.zeros((NWORLD, NRAY), dtype=int)
normal = wp.zeros((NWORLD, NRAY), dtype=wp.vec3)
allgroups = vec6(-1, -1, -1, -1, -1, -1)            # all -1: no group filtering

# without bodyexclude the origin is inside the robot's own cylinder
no_excl = wp.full(NRAY, -1, dtype=int)
mjw.rays(m, d, pnt, vec, allgroups, True, no_excl, dist, geomid, normal)
print("without bodyexclude, min range per world:", dist.numpy()[:4].min(1))

excl = wp.full(NRAY, robot, dtype=int)              # per-ray body to ignore
mjw.rays(m, d, pnt, vec, allgroups, True, excl, dist, geomid, normal)
r = dist.numpy()
g = geomid.numpy()
print("rays per call:", NWORLD * NRAY, "| output shape:", r.shape)
for w in range(4):
    k = int(np.argmin(np.where(r[w] < 0, np.inf, r[w])))
    name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, int(g[w, k]))
    print(f"world {w} pose {pose[w]}: min range {r[w, k]:.3f} m at body angle {k:3d} deg "
          f"({name}), misses {int((r[w] < 0).sum())}")
print("world 0 ranges at 0/90/180/270 deg:", r[0, [0, 90, 180, 270]])

if args.dump_json:
    geoms = []
    for i in range(mjm.ngeom):
        t = int(mjm.geom_type[i])
        if t in (mujoco.mjtGeom.mjGEOM_BOX, mujoco.mjtGeom.mjGEOM_CYLINDER) and i != 0:
            yaw = float(np.degrees(2 * np.arctan2(mjm.geom_quat[i, 3], mjm.geom_quat[i, 0])))
            geoms.append({"name": mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, i),
                          "type": "box" if t == mujoco.mjtGeom.mjGEOM_BOX else "cyl",
                          "pos": [round(float(v), 3) for v in mjm.geom_pos[i][:2]],
                          "size": [round(float(v), 3) for v in mjm.geom_size[i][:2]],
                          "yaw": round(yaw, 2)})
    out = {"nworld": 4, "nray": NRAY, "poses": pose[:4].tolist(),
           "ranges": [[round(float(v), 3) for v in r[w]] for w in range(4)],
           "gid": [g[w].tolist() for w in range(4)],
           "names": [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, i) for i in range(mjm.ngeom)],
           "geoms": [x for x in geoms if x["name"] != "robot"]}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Plot the lidar scans of Example 27.2 (polar: range against body angle).
try:
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 4, subplot_kw={"projection": "polar"}, figsize=(14, 3.6))
    for w in range(4):
        axs[w].plot(np.radians(np.arange(NRAY)), np.where(r[w] < 0, np.nan, r[w]))
        axs[w].set_title(f"world {w} pose {pose[w]}")
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## 3. Ray filters: `_ray_eliminate`
A geom is skipped if it is on `bodyexclude`, has alpha 0, is welded to the world while `flg_static=False`, or its group has a 0 in the `geomgroup` mask (all -1 = no filtering).

In [ ]:
# Example 27.3 - ex3_ray_filters.py
"""Example 27.3 - what a ray may hit: geomgroup, flg_static, bodyexclude and invisible geoms.

Five horizontal rays (one aimed at each object) are cast with different filters. The filter
rules are ray.py::_ray_eliminate: a geom is skipped if it is on bodyexclude, if its rgba (or
material) alpha is 0, if flg_static=False and its body is welded to the world, or if the
geomgroup mask is not all -1 and the entry for its group is 0.
Run: python ex3_ray_filters.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec6     # private API: the geomgroup mask type

FILTER_SCENE = """
<mujoco model="filters">
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1" group="0"/>
    <geom name="wall" type="box" pos="0 3 .5" size="3 .05 .5" group="1"/>
    <body name="crate" pos="-1 1 .2"><freejoint/>
      <geom name="crate" type="box" size=".2 .2 .2" group="2"/></body>
    <body name="probe" pos="0 1 .2"><freejoint/>
      <geom name="probe" type="sphere" size=".15" group="3"/></body>
    <body name="ghost" pos="1 1 .2"><freejoint/>
      <geom name="ghost" type="sphere" size=".15" rgba="1 1 1 0"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(FILTER_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=8, njmax=32)
mjw.kinematics(m, d)

# five rays from y = -1, z = 0.2, pointing +y, at x = -1 (crate), 0 (probe), 1 (ghost), 2, -2
xs = [-1.0, 0.0, 1.0, 2.0, -2.0]
pnt = wp.array([[wp.vec3(x, -1.0, 0.2) for x in xs]], dtype=wp.vec3)
vec = wp.array([[wp.vec3(0.0, 1.0, 0.0)] * len(xs)], dtype=wp.vec3)
dist = wp.zeros((1, len(xs)), dtype=float)
geomid = wp.zeros((1, len(xs)), dtype=int)
normal = wp.zeros((1, len(xs)), dtype=wp.vec3)


def cast(label, geomgroup, flg_static, bodyexclude):
    mjw.rays(m, d, pnt, vec, geomgroup, flg_static, wp.array(bodyexclude, dtype=int),
             dist, geomid, normal)
    names = []
    for g, r in zip(geomid.numpy()[0], dist.numpy()[0]):
        name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, int(g))
        names.append(f"{name}@{r:.2f}" if g >= 0 else "miss")
    print(f"{label:34s}", "  ".join(f"{n:11s}" for n in names))


print("geom groups:", dict(zip([mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, i)
                                for i in range(mjm.ngeom)], mjm.geom_group.tolist())))
print(f"{'ray x =':34s}", "  ".join(f"{x:<11.1f}" for x in xs))
NONE = [-1] * len(xs)
cast("all groups, static on", vec6(-1, -1, -1, -1, -1, -1), True, NONE)
cast("geomgroup excludes group 1 (wall)", vec6(1, 0, 1, 1, 1, 1), True, NONE)
cast("only group 3 (probe)", vec6(0, 0, 0, 1, 0, 0), True, NONE)
cast("flg_static=False", vec6(-1, -1, -1, -1, -1, -1), False, NONE)
crate = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "crate")
cast("bodyexclude crate on ray 0 only", vec6(-1, -1, -1, -1, -1, -1), True,
     [crate, -1, -1, -1, -1])
print("body_weldid (0 = welded to world):", mjm.body_weldid.tolist())

## 4. Rays through the scene BVH (`rc=`)
Without `rc`, `rays` is a tiled kernel (64 lanes per ray, `tile_argmin` over all geoms). With a render context it walks one BVH that groups boxes by world, starting from `rc.group_root[worldid]`. Refit after poses change. Note: with `rc`, only geoms in the context's `enabled_geom_groups` (default 0, 1, 2) can be hit.

In [ ]:
# Example 27.4 - ex4_rays_bvh.py
"""Example 27.4 - the same lidar, accelerated by the render context's scene BVH (rc=).

Without rc, mjw.rays launches the tiled kernel ray.py::_ray: each ray's block strides over
ALL geoms and reduces with wp.tile_argmin, so the cost per ray grows with ngeom. With
rc = mjw.create_render_context(...), ray.py::_ray_bvh walks the world-grouped BVH from
rc.group_root[worldid] and tests only the geoms whose boxes the ray enters.
The scene has 4 walls and NPILLAR random pillars. Timings are for THIS device; on a CPU they
are CPU numbers, not GPU numbers.
Run: python ex4_rays_bvh.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec6     # private API: the geomgroup mask type

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NRAY = 360
NPILLAR = 400 if ON_GPU else 200
NREP = 20 if ON_GPU else 3


def pillar_room(n, seed=0):
    rng = np.random.default_rng(seed)
    xy = rng.uniform(-1.8, 1.8, (n, 2))
    xy = xy[np.linalg.norm(xy, axis=1) > 0.5]            # keep the robot's spot free
    pillars = "".join(f'<geom type="cylinder" pos="{x:.3f} {y:.3f} .3" size=".03 .3"/>'
                      for x, y in xy)
    walls = "".join(f'<geom type="box" pos="{px} {py} .5" size="{sx} {sy} .5"/>'
                    for px, py, sx, sy in [(2, 0, .05, 2), (-2, 0, .05, 2),
                                           (0, 2, 2, .05), (0, -2, 2, .05)])
    return (f'<mujoco><worldbody><geom name="floor" type="plane" size="3 3 .1"/>{walls}{pillars}'
            f'<body name="robot" pos="0 0 .15"><freejoint/>'
            f'<geom type="cylinder" size=".15 .15"/></body></worldbody></mujoco>')


mjm = mujoco.MjModel.from_xml_string(pillar_room(NPILLAR))
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
robot = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "robot")
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 0] = np.linspace(-0.3, 0.3, NWORLD)              # each world's robot a bit further east
d.qpos.assign(qpos.astype(np.float32))
mjw.kinematics(m, d)

# the BVH lives in a render context; this model has no camera, so nothing is rendered
rc = mjw.create_render_context(mjm, nworld=NWORLD)
mjw.refit_bvh(m, d, rc)          # built from qpos0: refit to the current poses first
print(f"ngeom {mjm.ngeom} | bvh_ngeom {rc.bvh_ngeom} | BVH boxes {rc.lower.shape[0]}"
      f" (= nworld x bvh_ngeom) | nrender {rc.nrender}")
print("group_root (one BVH node per world):", rc.group_root.numpy()[:4])

ang = np.radians(np.arange(NRAY))
origin = d.xpos.numpy()[:, robot]                        # setup only: host copy is fine here
pnt = wp.array(np.repeat(origin[:, None, :], NRAY, 1), dtype=wp.vec3)
vec = wp.array(np.tile(np.stack([np.cos(ang), np.sin(ang), 0 * ang], 1), (NWORLD, 1, 1)),
               dtype=wp.vec3)
excl = wp.full(NRAY, robot, dtype=int)
allgroups = vec6(-1, -1, -1, -1, -1, -1)
out = {}
for label, ctx in (("brute force (rc=None)", None), ("scene BVH (rc=rc)", rc)):
    dist = wp.zeros((NWORLD, NRAY), dtype=float)
    gid = wp.zeros((NWORLD, NRAY), dtype=int)
    nrm = wp.zeros((NWORLD, NRAY), dtype=wp.vec3)
    mjw.rays(m, d, pnt, vec, allgroups, True, excl, dist, gid, nrm, rc=ctx)   # warm-up (JIT)
    wp.synchronize()
    t0 = time.perf_counter()
    for _ in range(NREP):
        mjw.rays(m, d, pnt, vec, allgroups, True, excl, dist, gid, nrm, rc=ctx)
    wp.synchronize()
    dt = (time.perf_counter() - t0) / NREP
    out[label] = (dist.numpy(), gid.numpy())
    print(f"{label:22s} {dt * 1e3:8.2f} ms per call  {NWORLD * NRAY / dt / 1e6:7.3f} Mrays/s"
          f"  [{wp.get_device()}]")
(d0, g0), (d1, g1) = out.values()
print("same geom hit on every ray:", bool((g0 == g1).all()),
      "| max |dist difference|:", float(np.abs(d0 - d1).max()))
print("world 0 min range:", round(float(d1[0][d1[0] > 0].min()), 3), "m")

## 5. The four-call render pipeline
`create_render_context` (once, keyword arguments only) -> `step`/`forward` -> `refit_bvh` -> `render` -> `get_rgb`/`get_depth`/`get_segmentation`. `cam_res` is (width, height); images come out as (nworld, height, width).

In [ ]:
# Example 27.5 - ex5_grid16.py
"""Example 27.5 - render every world at once: RGB, depth and segmentation in a grid.

The four calls: create_render_context (once) -> forward -> refit_bvh -> render, then
get_rgb / get_depth / get_segmentation unpack one camera into (nworld, H, W) arrays.
GPU: 16 worlds at 128x128; CPU: 4 worlds at 32x32.
Run: python ex5_grid16.py [--dump-png DIR]
  --dump-png DIR  writes ex5_rgb_grid.png and ex5_depth_grid.png (Rule D images for the page).
"""
import argparse
import os
import struct
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

RENDER_SCENE = """
<mujoco model="render_scene">
  <option timestep="0.01666"/>
  <worldbody>
    <light name="top" pos="0 0 2" dir="0 0 -1" directional="true"/>
    <camera name="camera" pos="0 2 0" quat="0 0 1 1" resolution="128 128"/>
    <geom name="floor" type="plane" pos="0 0 -.5" size="2 2 .1" rgba=".9 .9 .9 1"/>
    <body name="box_and_sphere" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
      <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
      <geom name="green_sphere" pos="-.2 -.2 -.2" size=".1" rgba="0 1 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


def tile(images, cols):
    """(n, H, W, C) -> one (rows*H, cols*W, C) image, world 0 top-left, row-major."""
    n, h, w, c = images.shape
    rows = n // cols
    return images[:rows * cols].reshape(rows, cols, h, w, c).transpose(0, 2, 1, 3, 4).reshape(
        rows * h, cols * w, c)


parser = argparse.ArgumentParser()
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 16 if ON_GPU else 4
W, H = (128, 128) if ON_GPU else (32, 32)
COLS = int(np.sqrt(NWORLD))

mjm = mujoco.MjModel.from_xml_string(RENDER_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=(W, H),   # (width, height)!
                               render_rgb=True, render_depth=True, render_seg=True)

qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 0] = np.linspace(0.0, 2 * np.pi, NWORLD, endpoint=False)   # a different swing per world
d.qpos.assign(qpos.astype(np.float32))
mjw.forward(m, d)                # geom_xpos/xmat and cam_xpos/xmat for the new qpos
mjw.refit_bvh(m, d, rc)          # boxes follow the new poses
mjw.render(m, d, rc)             # ONE launch: dim = (nworld, total_rays)

rgb = wp.zeros((NWORLD, H, W), dtype=wp.vec3)      # (nworld, height, width)
depth = wp.zeros((NWORLD, H, W), dtype=float)
seg = wp.zeros((NWORLD, H, W), dtype=wp.vec2i)
mjw.get_rgb(rc, 0, rgb)                           # camera_index 0 = first ACTIVE camera
mjw.get_depth(rc, 0, 3.0, depth)                  # depth / 3.0, clamped to [0, 1]
mjw.get_segmentation(rc, 0, seg)

img, dep, sg = rgb.numpy(), depth.numpy(), seg.numpy()
print(f"device {wp.get_device()} | nworld {NWORLD} | cam_res {rc.cam_res.numpy()[0]} (W, H)")
print("rgb", img.shape, img.dtype, "| depth", dep.shape, "| seg", sg.shape, sg.dtype)
print("mean RGB per world:", img.reshape(NWORLD, -1, 3).mean(1)[:4])
print("mean depth/3 per world:", dep.reshape(NWORLD, -1).mean(1)[:4])
for w in range(min(NWORLD, 4)):
    ids, counts = np.unique(sg[w, :, :, 0], return_counts=True)
    print(f"world {w} geom pixel counts:", dict(zip(ids.tolist(), counts.tolist())))

if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    grid = (np.clip(tile(img, COLS), 0, 1) * 255).astype(np.uint8)
    save_png(os.path.join(args.dump_png, "ex5_rgb_grid.png"), grid)
    dgrid = (np.clip(tile(dep[..., None], COLS), 0, 1) * 255).astype(np.uint8).repeat(3, 2)
    save_png(os.path.join(args.dump_png, "ex5_depth_grid.png"), dgrid)
    print("wrote", grid.shape, "grids to", args.dump_png)

In [ ]:
# Show the RGB and depth grids of Example 27.5 (img, dep and tile() come from the cell above).
try:
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(np.clip(tile(img, COLS), 0, 1)); axs[0].set_title("get_rgb, all worlds")
    axs[1].imshow(tile(dep[..., None], COLS)[..., 0], cmap="gray"); axs[1].set_title("get_depth / 3 m")
    for a in axs: a.axis("off")
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

In [ ]:
# Optional comparison: the same world-0 frame from MuJoCo's own OpenGL renderer (EGL), next to
# mjw.get_rgb. Uses mjm, d, img, W and H from Example 27.5 above. The two images will not match
# pixel for pixel: the batch renderer is a separate ray tracer with its own shading.
try:
    import matplotlib.pyplot as plt
    mjd = mujoco.MjData(mjm)
    mjd.qpos[:] = d.qpos.numpy()[0]
    mujoco.mj_forward(mjm, mjd)
    renderer = mujoco.Renderer(mjm, height=H, width=W)
    renderer.update_scene(mjd, camera="camera")
    gl = renderer.render()
    renderer.close()
    fig, axs = plt.subplots(1, 2, figsize=(8, 4))
    axs[0].imshow(gl); axs[0].set_title("mujoco.Renderer (OpenGL / EGL)")
    axs[1].imshow(np.clip(img[0], 0, 1)); axs[1].set_title("mjw.get_rgb, world 0")
    for a in axs:
        a.axis("off")
    plt.show()
except Exception as e:  # no EGL / OpenGL on this runtime: the batch renderer above does not need it
    print("OpenGL comparison skipped:", type(e).__name__, e)

## 6. Why `refit_bvh`: the stale-BVH demonstration
The context's BVH is built at `qpos0`. Move the sphere without a refit and it vanishes (16 -> 0 -> 18 sphere pixels).

In [ ]:
# Example 27.7 - ex7_stale_bvh.py
"""Example 27.7 - why refit_bvh: a moved sphere vanishes until the BVH is refit.

The render context builds its BVH from the model's initial pose. After the sphere moves
1 m to the side, render() still culls rays against the OLD bounding box, so the sphere is
not found at its new place (and its old box is empty). refit_bvh recomputes the boxes from
d.geom_xpos / d.geom_xmat and the sphere comes back.
Run: python ex7_stale_bvh.py [--dump-png DIR]
  --dump-png DIR  writes the three frames as ex7_frame{0,1,2}.png (Rule D flipbook).
"""
import argparse
import os
import struct
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BALL = """
<mujoco model="stale_bvh">
  <worldbody>
    <light dir="0 0 -1" pos="0 0 3" directional="true"/>
    <camera name="cam" pos="0 -3 0.5" xyaxes="1 0 0 0 0 1"/>
    <geom name="floor" type="plane" size="3 3 .1" rgba=".3 .3 .3 1"/>
    <body name="ball" pos="0 0 0.5"><freejoint/>
      <geom name="ball" type="sphere" size=".2" rgba="1 0 0 1"/></body>
  </worldbody>
</mujoco>
"""


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


parser = argparse.ArgumentParser()
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

W, H = 40, 30
mjm = mujoco.MjModel.from_xml_string(BALL)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
mjw.forward(m, d)
rc = mjw.create_render_context(mjm, nworld=1, cam_res=(W, H), render_rgb=True,
                               render_depth=False, render_seg=True)
ball = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_GEOM, "ball")
rgb = wp.zeros((1, H, W), dtype=wp.vec3)
frames = []


def frame(label):
    mjw.render(m, d, rc)
    seg = rc.seg_data.numpy()[0]
    mjw.get_rgb(rc, 0, rgb)
    frames.append((np.clip(rgb.numpy()[0], 0, 1) * 255).astype(np.uint8))
    lo, hi = rc.lower.numpy()[ball], rc.upper.numpy()[ball]
    print(f"{label:28s} sphere pixels {int((seg[:, 0] == ball).sum()):3d} | "
          f"BVH box x [{lo[0]:+.2f}, {hi[0]:+.2f}]")


frame("start (BVH built at qpos0):")
q = d.qpos.numpy()
q[0, 0] = 1.0                       # move the free body 1 m along +x
d.qpos.assign(q)
mjw.forward(m, d)                   # geom_xpos is now x = 1 ...
print("sphere geom_xpos after forward:", d.geom_xpos.numpy()[0, ball])
frame("moved, NO refit:")           # ... but the BVH box is still at x = 0
mjw.refit_bvh(m, d, rc)
frame("moved, after refit_bvh:")

if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    for i, f in enumerate(frames):
        save_png(os.path.join(args.dump_png, f"ex7_frame{i}.png"), f)
    print("wrote 3 frames to", args.dump_png)

In [ ]:
# The three frames of Example 27.7: start, moved without refit, after refit_bvh.
try:
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.2))
    for a, f, t in zip(axs, frames, ["start", "moved, NO refit", "after refit_bvh"]):
        a.imshow(f, interpolation="nearest"); a.set_title(t); a.axis("off")
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## 7. Every option, geom groups and `cam_active`
The option table comes from `inspect.signature`, so it matches the installed version. Geoms in group 3+ are invisible with the default `enabled_geom_groups`. `camera_index` counts *active* cameras (`rc.cam_id_map` maps back).

In [ ]:
# Example 27.12 - ex12_options_and_groups.py
"""Example 27.12 - every create_render_context option, geom groups, and cam_active by name.

Part 1 prints the option table straight from inspect.signature (so it always matches your
installed version). Part 2: a geom in group 3 is not in the BVH with the default
enabled_geom_groups=[0, 1, 2], so it is invisible; [0, 1, 2, 3] shows it. Part 3: cam_active
selects cameras by name; camera_index in get_rgb is the position in that ACTIVE list.
Run: python ex12_options_and_groups.py
"""
import inspect

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

GROUPS = """
<mujoco model="groups">
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1" directional="true"/>
    <camera name="front" pos="0 -3 .5" xyaxes="1 0 0 0 0 1"/>
    <camera name="side" pos="3 0 .5" xyaxes="0 1 0 0 0 1"/>
    <geom name="floor" type="plane" size="3 3 .1" group="0"/>
    <geom name="visual_ball" type="sphere" pos="-.5 0 .5" size=".25" group="1" rgba="1 .3 .3 1"/>
    <geom name="collision_box" type="box" pos=".5 0 .5" size=".2 .2 .2" group="3"
          rgba=".3 .3 1 1"/>
  </worldbody>
</mujoco>
"""

# Part 1: the options table, from the installed library itself
sig = inspect.signature(mjw.create_render_context)
print(f"create_render_context: {len(sig.parameters)} parameters")
for name, p in sig.parameters.items():
    default = "(required)" if p.default is inspect.Parameter.empty else repr(p.default)
    print(f"  {name:25s} = {default}")

mjm = mujoco.MjModel.from_xml_string(GROUPS)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
mjw.forward(m, d)
box = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_GEOM, "collision_box")


def box_pixels(**opts):
    rc = mjw.create_render_context(mjm, nworld=1, cam_res=(32, 32), render_rgb=True,
                                   render_depth=False, render_seg=True, **opts)
    mjw.refit_bvh(m, d, rc)
    mjw.render(m, d, rc)
    seg = rc.seg_data.numpy()[0]
    return rc, int((seg[:, 0] == box).sum())


# Part 2: geom groups (keyword arguments only, always)
rc, n = box_pixels(cam_active=["front"])
print("\ngeom groups:", mjm.geom_group.tolist(), "| bvh_ngeom with default groups:", rc.bvh_ngeom)
print("group-3 box pixels, enabled_geom_groups=[0, 1, 2]   :", n)
rc, n = box_pixels(cam_active=["front"], enabled_geom_groups=[0, 1, 2, 3])
print("group-3 box pixels, enabled_geom_groups=[0, 1, 2, 3]:", n, "| bvh_ngeom:", rc.bvh_ngeom)

# Part 3: cam_active by name -> camera_index counts ACTIVE cameras only
for active in (None, ["side"], ["side", "front"]):
    rc, n = box_pixels(cam_active=active, enabled_geom_groups=[0, 1, 2, 3])
    names = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_CAMERA, int(c))
             for c in rc.cam_id_map.numpy()]
    print(f"cam_active={active!s:17s} nrender {rc.nrender} | cam_id_map "
          f"{rc.cam_id_map.numpy().tolist()} -> camera_index 0 is {names[0]!r}"
          f" | box px (all cams) {n}")

## 8. Camera modes and the ordering rule
`camlight` (inside every forward/step) writes `d.cam_xpos`/`d.cam_xmat` from the MJCF modes. A camera you set by hand goes after `step` and before `refit_bvh`/`render`, every frame.

In [ ]:
# Example 27.14 - ex14_camera_modes.py
"""Example 27.14 - camera modes follow a falling body; the ordering rule for manual cameras.

Four cameras watch a box thrown sideways: 'fixed' (world body), 'track' (moves with the box,
keeps its world orientation), 'trackcom' (moves with the subtree centre of mass) and 'target'
(fixed position, mode targetbody: turns to face the box). smooth.py::camlight recomputes
d.cam_xpos / d.cam_xmat inside every forward/step. So a camera you set by hand must be written
AFTER step and BEFORE refit_bvh/render, every frame; a write before step is overwritten.
Run: python ex14_camera_modes.py [--dump-json PATH]
  --dump-json PATH  writes the box and camera poses (world 0) for the page's 3-D widget.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

CAM_MODES = """
<mujoco model="camera_modes">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 4" dir="0 0 -1" directional="true"/>
    <camera name="fixed" pos="0 -4 1.5" xyaxes="1 0 0 0 .35 .94"/>
    <camera name="target" mode="targetbody" target="box" pos="-2 -2 1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="box" pos="-1 0 1.5"><freejoint/>
      <geom name="box" type="box" size=".15 .15 .15" mass="1" rgba=".9 .3 .2 1"/>
      <geom name="weight" type="sphere" pos=".3 0 0" size=".08" mass="2" rgba=".2 .2 .9 1"/>
      <camera name="track" mode="track" pos="0 -1.5 .5" xyaxes="1 0 0 0 .3 .95"/>
      <camera name="trackcom" mode="trackcom" pos="0 -1.5 .5" xyaxes="1 0 0 0 .3 .95"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD, W, H = 4, 24, 24
NFRAME, NSUB = 5, 20                               # 5 frames, 0.1 s apart
mjm = mujoco.MjModel.from_xml_string(CAM_MODES)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
cams = [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_CAMERA, i) for i in range(mjm.ncam)]
print("cameras:", cams, "| cam_mode:", mjm.cam_mode.tolist(), "(0 fixed, 1 track, "
      "2 trackcom, 3 targetbody)")
qvel = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
qvel[:, 0] = np.linspace(1.0, 2.5, NWORLD)        # each world throws the box at its own speed
qvel[:, 4] = 4.0                                   # and spins it about y (COM != body origin)
d.qvel.assign(qvel)
mjw.forward(m, d)

track, trackcom, target = cams.index("track"), cams.index("trackcom"), cams.index("target")
frames = []
for f in range(NFRAME):
    for i in range(NSUB):
        mjw.step(m, d)
        if args.dump_json and i % 5 == 4:          # 4 snapshots per frame for the widget
            frames.append((d.xpos.numpy()[0, 1].copy(), d.cam_xpos.numpy()[0].copy(),
                           d.cam_xmat.numpy()[0].copy(), d.xmat.numpy()[0, 1].copy(),
                           d.subtree_com.numpy()[0, 1].copy()))
    xpos = d.cam_xpos.numpy()
    fwd = -d.cam_xmat.numpy()[:, target][:, :, 2]  # camera looks along -z of cam_xmat
    print(f"t={d.time.numpy()[0]:.2f}s track x per world {xpos[:, track, 0]} | trackcom x w0 "
          f"{xpos[0, trackcom, 0]:+.3f} | target fwd w0 {fwd[0]}")

# the ordering rule: write the 'fixed' camera by hand, then render
rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=(W, H), render_rgb=True,
                               render_depth=False, render_seg=False, cam_active=["fixed"])
rgb = wp.zeros((NWORLD, H, W), dtype=wp.vec3)
fixed = cams.index("fixed")


def render_fixed():
    mjw.refit_bvh(m, d, rc)
    mjw.render(m, d, rc)
    mjw.get_rgb(rc, 0, rgb)
    return rgb.numpy().copy()


def write_top_view():
    pos, mat = d.cam_xpos.numpy(), d.cam_xmat.numpy()
    pos[:, fixed] = [0.0, 0.0, 5.0]                # straight above, looking down (-z)
    mat[:, fixed] = np.eye(3)                      # columns = camera x, y, z axes in world
    d.cam_xpos.assign(pos)
    d.cam_xmat.assign(mat)


write_top_view()                                   # WRONG place: before step ...
mjw.step(m, d)                                     # ... camlight (in forward) overwrites it
print("written before step -> after step cam_xpos[0, fixed] =", d.cam_xpos.numpy()[0, fixed])
default = render_fixed()                           # the MJCF view of this exact state
write_top_view()                                   # RIGHT place: after step, before render
right = render_fixed()
changed = np.any(np.abs(right - default) > 1e-3, axis=-1).reshape(NWORLD, -1).mean(1)
print("written after step  -> cam_xpos[0, fixed] =", d.cam_xpos.numpy()[0, fixed],
      "| fraction of pixels changed per world:", changed.round(2))

if args.dump_json:
    out = {"cams": cams, "modes": mjm.cam_mode.tolist(), "dt": 5 * float(mjm.opt.timestep),
           "frames": [{"box": [round(float(v), 3) for v in b],
                       "com": [round(float(v), 3) for v in c],
                       "rot": [round(float(v), 3) for v in bm.ravel()],
                       "pos": [[round(float(v), 3) for v in p] for p in cp],
                       "fwd": [[round(float(-v), 3) for v in R[:, 2]] for R in cm],
                       "up": [[round(float(v), 3) for v in R[:, 1]] for R in cm]}
                      for b, cp, cm, bm, c in frames]}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 9. Output buffers and their layout
One row per world; cameras laid end to end; `rgb_adr`/`depth_adr`/`seg_adr` give each camera's start (-1 = no such output); pixels row-major. RGB is packed `0xAARRGGBB`.

In [ ]:
# Example 27.6 - ex6_buffer_layout.py
"""Example 27.6 - the flat output buffers: two cameras, different resolutions and outputs.

cam0 renders 32x24 RGB + depth + seg, cam1 16x16 RGB + seg (no depth). Every world owns one
row of rc.rgb_data (uint32 0xAARRGGBB) holding all cameras' pixels back to back; camera c
starts at rc.rgb_adr[c] and its pixels are row-major (py * W + px). A camera without an output
has adr -1 there. The manual unpack below gives exactly what mjw.get_rgb returns.
Run: python ex6_buffer_layout.py [--dump-json PATH]
  --dump-json PATH  writes the real packed pixels, depth and seg ids for the page's mosaic.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

TWO_CAMS = """
<mujoco model="two_cams">
  <worldbody>
    <light name="top" pos="0 0 2" dir="0 0 -1" directional="true"/>
    <camera name="front" pos="0 2 0" quat="0 0 1 1"/>
    <camera name="above" pos="0 0 1.6" xyaxes="1 0 0 0 1 0"/>
    <geom name="floor" type="plane" pos="0 0 -.5" size="2 2 .1" rgba=".9 .9 .9 1"/>
    <body name="box_and_sphere" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
      <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
      <geom name="green_sphere" pos="-.2 -.2 -.2" size=".1" rgba="0 1 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 2
RES = [(32, 24), (16, 16)]                       # (width, height) per active camera

mjm = mujoco.MjModel.from_xml_string(TWO_CAMS)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=RES,
                               render_rgb=[True, True], render_depth=[True, False],
                               render_seg=[True, True])
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 0] = [0.0, 1.2]
d.qpos.assign(qpos.astype(np.float32))
mjw.forward(m, d)
mjw.refit_bvh(m, d, rc)
mjw.render(m, d, rc)

print("nrender", rc.nrender, "| total_rays", rc.total_rays, "=", " + ".join(
    str(w * h) for w, h in RES), "| cam_id_map", rc.cam_id_map.numpy())
print("cam_res (W, H):", rc.cam_res.numpy().tolist())
print("rgb_adr", rc.rgb_adr.numpy(), "depth_adr", rc.depth_adr.numpy(),
      "seg_adr", rc.seg_adr.numpy())
print("rgb_data", rc.rgb_data.shape, rc.rgb_data.dtype.__name__, "| depth_data",
      rc.depth_data.shape, "| seg_data", rc.seg_data.shape, "| ray", rc.ray.shape)

packed = rc.rgb_data.numpy()                     # (nworld, 1024) uint32
for px, py in [(0, 0), (16, 12)]:                # a background pixel and the image centre
    p = int(packed[0, py * 32 + px])             # cam0 starts at rgb_adr[0] = 0, W = 32
    print(f"world 0, cam0 pixel (px={px}, py={py}): 0x{p:08X} -> A={p >> 24} "
          f"R={(p >> 16) & 255} G={(p >> 8) & 255} B={p & 255}")

for cam, (w, h) in enumerate(RES):
    adr = int(rc.rgb_adr.numpy()[cam])
    block = packed[:, adr:adr + w * h].reshape(NWORLD, h, w)       # row-major: [py, px]
    manual = np.stack([(block >> 16) & 255, (block >> 8) & 255, block & 255], -1) / 255.0
    out = wp.zeros((NWORLD, h, w), dtype=wp.vec3)
    mjw.get_rgb(rc, cam, out)
    print(f"cam{cam}: manual unpack == get_rgb:", bool(np.allclose(manual, out.numpy())),
          f"| block [{adr}:{adr + w * h}] -> (nworld, H, W) = {manual.shape[:3]}")

if args.dump_json:
    dep = rc.depth_data.numpy()
    sg = rc.seg_data.numpy()
    out = {"nworld": NWORLD, "res": RES, "rgb_adr": rc.rgb_adr.numpy().tolist(),
           "depth_adr": rc.depth_adr.numpy().tolist(), "seg_adr": rc.seg_adr.numpy().tolist(),
           "total_rays": rc.total_rays,
           "geom_names": [mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, i)
                          for i in range(mjm.ngeom)],
           "rgb": packed.astype(np.int64).tolist(),
           "depth": [[round(float(v), 3) for v in row] for row in dep],
           "seg": [[int(v[0]) for v in row] for row in sg],
           "segtype": [[int(v[1]) for v in row] for row in sg]}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 10. Shadows and supersampling
Both are creation options. A blocked shadow ray scales the light by `shadow_light_fraction` (0.3). `samples_per_pixel=n` renders n x n passes and averages.

In [ ]:
# Example 27.9 - ex9_shading_options.py
"""Example 27.9 - shadows and supersampling are creation-time options of the render context.

Same scene, three contexts: plain, use_shadows=True, and samples_per_pixel=2. A shadow ray
(any-hit, from the hit point toward the light) that is blocked scales that light by
shadow_light_fraction (default 0.3). samples_per_pixel=n casts n x n sub-pixel rays and
averages the colour; the precomputed ray buffer grows to total_rays * n * n.
Run: python ex9_shading_options.py [--dump-png DIR]
  --dump-png DIR  writes ex9_plain.png, ex9_shadow.png and ex9_ssaa.png.
"""
import argparse
import os
import struct
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

SHADOW_SCENE = """
<mujoco model="shadow_scene">
  <visual><headlight active="0"/></visual>
  <worldbody>
    <light name="sun" pos="0 0 3" dir=".6 0 -1" directional="true" castshadow="true"/>
    <camera name="down" pos="0 0 3" xyaxes="1 0 0 0 1 0" fovy="50"/>
    <geom name="floor" type="plane" size="3 3 .1" rgba=".8 .8 .8 1"/>
    <geom name="block" type="box" pos="-.3 0 .4" size=".2 .2 .4" euler="0 0 20"
          rgba=".2 .4 .9 1"/>
  </worldbody>
</mujoco>
"""


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


parser = argparse.ArgumentParser()
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

W = H = 32
mjm = mujoco.MjModel.from_xml_string(SHADOW_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
mjw.forward(m, d)
images = {}
for label, opts in (("plain", {}), ("shadow", {"use_shadows": True}),
                    ("ssaa", {"samples_per_pixel": 2})):
    rc = mjw.create_render_context(mjm, nworld=1, cam_res=(W, H), render_rgb=True,
                                   render_depth=False, render_seg=True, **opts)
    mjw.refit_bvh(m, d, rc)
    mjw.render(m, d, rc)
    out = wp.zeros((1, H, W), dtype=wp.vec3)
    mjw.get_rgb(rc, 0, out)
    images[label] = out.numpy()[0]
    seg = rc.seg_data.numpy()[0, :W * H].reshape(H, W, 2)
    print(f"{label:6s} total_rays {rc.total_rays} | ray buffer {rc.ray.shape[0]:5d} | "
          f"aa_accum {rc.aa_accum.shape}")

floor = seg[..., 0] == 0                            # seg is identical in all three contexts
plain, shadow, ssaa = images["plain"], images["shadow"], images["ssaa"]
changed = np.any(np.abs(shadow - plain) > 1e-6, axis=-1)
in_shadow = changed & floor
ratio = shadow[in_shadow, 0] / plain[in_shadow, 0]
print("pixels changed by use_shadows:", int(changed.sum()), "of", W * H,
      "| shadowed floor pixels:", int(in_shadow.sum()))
print("shadowed / lit floor red channel:", np.round(np.median(ratio), 3),
      "(shadow_light_fraction = 0.3)")
edge = np.any(np.abs(ssaa - plain) > 1e-6, axis=-1)
print("pixels changed by samples_per_pixel=2:", int(edge.sum()),
      "| distinct colours plain vs 2x2:", len(np.unique(plain.reshape(-1, 3), axis=0)),
      len(np.unique(ssaa.reshape(-1, 3), axis=0)))

if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    for k, v in images.items():
        save_png(os.path.join(args.dump_png, f"ex9_{k}.png"),
                 (np.clip(v, 0, 1) * 255).astype(np.uint8))
    print("wrote ex9_plain/shadow/ssaa.png to", args.dump_png)

In [ ]:
# Example 27.9 side by side.
try:
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    for a, (k, v) in zip(axs, images.items()):
        a.imshow(np.clip(v, 0, 1), interpolation="nearest"); a.set_title(k); a.axis("off")
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## 11. Depth and segmentation
Depth is planar (z-depth): `dist * -d_cam.z`. `get_depth` divides by `depth_scale` and clamps to [0, 1]; read `rc.depth_data` for metres. Segmentation pixels are `(objid, objtype)` with 5 = geom, 9 = flex, (-1, -1) = background. (The first run compiles the flex kernels, which takes a while.)

In [ ]:
# Example 27.8 - ex8_depth_and_seg.py
"""Example 27.8 - depth is planar (z-depth), get_depth clamps, and seg holds (objid, objtype).

A camera 2 m above a floor looks straight down. Planar depth is the distance along the
optical axis, so every floor pixel reads 2.0 m, even in the corners where the ray itself is
longer. We check that with mjw.rays along the corner pixel's ray (rc.ray holds the precomputed
camera-frame directions). get_depth(rc, cam, scale, out) returns clamp(depth / scale, 0, 1):
fine for a picture, lossy for metric work. A small flex cloth shows objtype 9 (mjOBJ_FLEX).
Run: python ex8_depth_and_seg.py [--dump-png DIR]
  --dump-png DIR  writes ex8_depth.png and ex8_seg.png (colorized ids).
"""
import argparse
import os
import struct
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec6     # private API: the geomgroup mask type

np.set_printoptions(precision=4, suppress=True)

TOP_DOWN = """
<mujoco model="top_down">
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1" directional="true"/>
    <camera name="down" pos="0 0 2" xyaxes="1 0 0 0 1 0" fovy="60"/>
    <geom name="floor" type="plane" size="3 3 .1" rgba=".8 .8 .8 1"/>
    <geom name="block" type="box" pos=".4 .3 .25" size=".2 .2 .25" rgba=".2 .4 .9 1"/>
    <geom name="ball" type="sphere" pos="-.4 -.2 .3" size=".3" rgba=".9 .3 .2 1"/>
    <flexcomp name="cloth" type="grid" count="4 4 1" spacing=".12 .12 .12" pos="-.3 .45 .6"
              dim="2" radius=".01" rgba=".2 .8 .3 1">
      <edge equality="true"/>
    </flexcomp>
  </worldbody>
</mujoco>
"""


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


parser = argparse.ArgumentParser()
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

W = H = 32
mjm = mujoco.MjModel.from_xml_string(TOP_DOWN)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=64, njmax=256)
rc = mjw.create_render_context(mjm, nworld=1, cam_res=(W, H), render_rgb=False,
                               render_depth=True, render_seg=True)
mjw.forward(m, d)
mjw.refit_bvh(m, d, rc)
mjw.render(m, d, rc)

metric = rc.depth_data.numpy()[0, :W * H].reshape(H, W)        # metres, 0 = miss
clamped = wp.zeros((1, H, W), dtype=float)
mjw.get_depth(rc, 0, 1.0, clamped)                              # clamp(depth / 1.0, 0, 1)
print(f"znear {rc.znear:.4f} m, zfar {rc.zfar:.2f} m (vis.map.znear/zfar x stat.extent)")
print("centre pixel: metric depth", round(float(metric[16, 16]), 4),
      "| get_depth(scale=1.0)", round(float(clamped.numpy()[0, 16, 16]), 4))
print("corner pixel: metric depth", round(float(metric[0, 0]), 4),
      "| floor pixels min/max depth", metric[metric > 1.99].min().round(4),
      metric[metric > 1.99].max().round(4))

# the same corner ray through the public ray API: Euclidean length vs planar depth
loc = rc.ray.numpy()[0]                                         # pixel (0,0), camera frame
R = d.cam_xmat.numpy()[0, 0]
pnt = wp.array([[d.cam_xpos.numpy()[0, 0]]], dtype=wp.vec3)
vec = wp.array([[R @ loc]], dtype=wp.vec3)
dist, gid, _ = mjw.ray(m, d, pnt, vec, geomgroup=vec6(-1, -1, -1, -1, -1, -1))
print("corner ray length", round(float(dist.numpy()[0, 0]), 4), "x cos(theta)",
      round(float(-loc[2]), 4), "=", round(float(dist.numpy()[0, 0] * -loc[2]), 4))

seg = rc.seg_data.numpy()[0, :W * H].reshape(H, W, 2)
pairs, counts = np.unique(seg.reshape(-1, 2), axis=0, return_counts=True)
names = {5: "mjOBJ_GEOM", 9: "mjOBJ_FLEX", -1: "background"}
for (oid, otype), c in zip(pairs.tolist(), counts.tolist()):
    if otype == 5:
        label = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, oid)
    elif otype == 9:
        label = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_FLEX, oid)
    else:
        label = "-"
    print(f"(objid {oid:2d}, objtype {otype:2d} {names[otype]:10s}) {label:6s} {c:4d} px")
print("mjOBJ_GEOM =", int(mujoco.mjtObj.mjOBJ_GEOM),
      "| mjOBJ_FLEX =", int(mujoco.mjtObj.mjOBJ_FLEX))

if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    dep = (np.clip(1.0 - (metric - 1.2) / 0.9, 0, 1) * 255).astype(np.uint8)   # near = bright
    dep[metric == 0] = 0
    save_png(os.path.join(args.dump_png, "ex8_depth.png"), dep[..., None].repeat(3, 2))
    rng = np.random.default_rng(42)                 # same idea as record.py's seg palette
    palette = rng.integers(40, 255, (16, 3)).astype(np.uint8)
    key = np.where(seg[..., 0] < 0, 0, seg[..., 0] + 1 + 8 * (seg[..., 1] == 9))
    img = palette[key % 16]
    img[seg[..., 0] < 0] = 0
    save_png(os.path.join(args.dump_png, "ex8_seg.png"), img)
    print("wrote ex8_depth.png and ex8_seg.png to", args.dump_png)

## 12. Visual domain randomization
Batch Model fields with `put_model(batch_sizes=...)`. Colours work directly; a per-world field of view needs `use_precomputed_rays=False`.

In [ ]:
# Example 27.10 - ex10_visual_dr.py
"""Example 27.10 - visual domain randomization: per-world colours and per-world field of view.

put_model(mjm, batch_sizes={...}) gives geom_rgba and cam_fovy a leading per-world axis; the
megakernel reads them at worldid % shape[0]. Colours just work. The field of view only works
with use_precomputed_rays=False: precomputed rays are built once from the MjModel's cam_fovy,
so every world would keep the same camera.
Run: python ex10_visual_dr.py [--dump-png DIR]
  --dump-png DIR  writes ex10_precomputed.png and ex10_dynamic.png (3 worlds side by side).
"""
import argparse
import os
import struct
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

DR_SCENE = """
<mujoco model="visual_dr">
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1" directional="true"/>
    <camera name="c" pos="0 -3 0.5" xyaxes="1 0 0 0 0 1" fovy="45"/>
    <geom name="floor" type="plane" size="3 3 .1" rgba=".3 .3 .3 1"/>
    <geom name="s" type="sphere" pos="0 0 .5" size=".3" rgba="1 0 0 1"/>
  </worldbody>
</mujoco>
"""


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


parser = argparse.ArgumentParser()
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

NWORLD, W, H = 3, 32, 32
mjm = mujoco.MjModel.from_xml_string(DR_SCENE)
m = mjw.put_model(mjm, batch_sizes={"cam_fovy": NWORLD, "geom_rgba": NWORLD})
print("cam_fovy", m.cam_fovy.shape, "| geom_rgba", m.geom_rgba.shape, "(leading axis = worlds)")

fovy = m.cam_fovy.numpy()
fovy[:, 0] = [20.0, 45.0, 90.0]
m.cam_fovy.assign(fovy)
rgba = m.geom_rgba.numpy()
rgba[:, 1] = [[1, 0, 0, 1], [0, 1, 0, 1], [0, 0, 1, 1]]      # sphere colour per world
m.geom_rgba.assign(rgba)

d = mjw.make_data(mjm, nworld=NWORLD)
mjw.forward(m, d)
strips = {}
for pre in (True, False):
    rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=(W, H), render_rgb=True,
                                   render_depth=False, render_seg=True,
                                   use_precomputed_rays=pre)
    mjw.refit_bvh(m, d, rc)
    mjw.render(m, d, rc)
    seg = rc.seg_data.numpy()
    rgb = wp.zeros((NWORLD, H, W), dtype=wp.vec3)
    mjw.get_rgb(rc, 0, rgb)
    strips[pre] = rgb.numpy()
    label = "use_precomputed_rays=True " if pre else "use_precomputed_rays=False"
    counts = [int((seg[w, :, 0] == 1).sum()) for w in range(NWORLD)]
    print(f"{label}: sphere pixels per world {counts} | ray buffer {rc.ray.shape[0]}")
print("fovy per world:", fovy[:, 0].tolist(), "| centre colour per world:",
      strips[False][:, 16, 16].astype(float).round(2).tolist())

if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    for pre, name in ((True, "precomputed"), (False, "dynamic")):
        row = np.concatenate([np.pad(x, ((1, 1), (1, 1), (0, 0)), constant_values=1.0)
                              for x in strips[pre]], axis=1)
        save_png(os.path.join(args.dump_png, f"ex10_{name}.png"),
                 (np.clip(row, 0, 1) * 255).astype(np.uint8))
    print("wrote ex10_precomputed.png and ex10_dynamic.png to", args.dump_png)

In [ ]:
# Example 27.10: the three worlds with precomputed vs dynamic rays.
try:
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(2, 3, figsize=(9, 6))
    for row, pre in enumerate((True, False)):
        for w in range(NWORLD):
            axs[row, w].imshow(np.clip(strips[pre][w], 0, 1), interpolation="nearest")
            axs[row, w].set_title(f"fovy {fovy[w, 0]:.0f}, precomputed={pre}", fontsize=9)
            axs[row, w].axis("off")
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## 13. Gaussian splats with a splat set per world
`splat_adr` cuts the arrays into groups, `splat_group_id[w]` picks a group per world. Splats are composited front to back and hidden by solid geometry. After editing `rc.splat_*`, call `mjw.refit_splat_bvh(rc)`.

In [ ]:
# Example 27.13 - ex13_splats.py
"""Example 27.13 - Gaussian splats next to meshes, with a different splat set per world.

N random 3-D Gaussians are made in NumPy (no PLY file needed): group 0 is a warm cloud, group 1
a cool cloud. splat_adr = [0, n0, n0 + n1] cuts the arrays into groups and
splat_group_id[w] says which group world w sees. Splats are composited front to back along
each ray (render.py::shade_splats); solid geometry in front hides them.
Run: python ex13_splats.py [--dump-png DIR]
  --dump-png DIR  writes ex13_splats.png (world 0 | world 1, without and with splats).
"""
import argparse
import os
import struct
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

SPLAT_SCENE = """
<mujoco model="splat_scene">
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1" directional="true"/>
    <camera name="cam" pos="0 -3 .6" xyaxes="1 0 0 0 0 1"/>
    <geom name="floor" type="plane" size="3 3 .1" rgba=".5 .5 .5 1"/>
    <geom name="post" type="cylinder" pos=".25 -1 .3" size=".08 .3" rgba=".9 .9 .2 1"/>
  </worldbody>
</mujoco>
"""


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


def cloud(rng, n, centre, rgb):
    pos = centre + rng.normal(0, 0.18, (n, 3))
    q = rng.normal(size=(n, 4))
    q /= np.linalg.norm(q, axis=1, keepdims=True)               # unit quaternions (w, x, y, z)
    scale = rng.uniform(0.03, 0.09, (n, 3))                     # standard deviations [m]
    rgba = np.concatenate([np.clip(rgb + rng.normal(0, 0.08, (n, 3)), 0, 1),
                           rng.uniform(0.4, 0.9, (n, 1))], 1)   # colour + opacity
    return pos, q, scale, rgba


parser = argparse.ArgumentParser()
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
N = 2000 if ON_GPU else 150                                     # splats per group
NWORLD, W, H = 2, 32, 32
rng = np.random.default_rng(0)
g0 = cloud(rng, N, np.array([-0.3, 0.0, 0.6]), np.array([1.0, 0.5, 0.1]))
g1 = cloud(rng, N, np.array([0.3, 0.0, 0.6]), np.array([0.1, 0.5, 1.0]))
pos, rot, scale, rgba = (np.concatenate([a, b]).astype(np.float32) for a, b in zip(g0, g1))
splat_adr = np.array([0, N, 2 * N], dtype=np.int32)             # group k = [adr[k], adr[k+1])
splat_group_id = np.array([0, 1], dtype=np.int32)               # world 0 -> group 0, world 1 -> 1

mjm = mujoco.MjModel.from_xml_string(SPLAT_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
mjw.forward(m, d)
images = {}
for label, kw in (("no splats", {}),
                  ("splats", dict(splat_position=pos, splat_rotation=rot, splat_scale=scale,
                                  splat_rgba=rgba, splat_adr=splat_adr,
                                  splat_group_id=splat_group_id))):
    rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=(W, H), render_rgb=True,
                                   render_depth=True, render_seg=True, **kw)
    mjw.refit_bvh(m, d, rc)
    mjw.render(m, d, rc)
    img = wp.zeros((NWORLD, H, W), dtype=wp.vec3)
    mjw.get_rgb(rc, 0, img)
    images[label] = img.numpy()
    roots = rc.splat_group_root.numpy().tolist() if rc.splat_count else "unused"  # BVH node/world
    print(f"{label:9s}: splat_count {rc.splat_count:4d} | splat_group_root {roots!s:10s} | "
          f"mean RGB w0 {img.numpy()[0].mean((0, 1))} w1 {img.numpy()[1].mean((0, 1))}")

diff = np.any(np.abs(images["splats"] - images["no splats"]) > 1 / 255, axis=-1)
print("pixels changed by splats per world:", diff.reshape(NWORLD, -1).sum(1).tolist())
left, right = diff[:, :, :W // 2].sum((1, 2)), diff[:, :, W // 2:].sum((1, 2))
print("changed pixels left/right half: world 0", [int(left[0]), int(right[0])],
      "| world 1", [int(left[1]), int(right[1])])
seg = rc.seg_data.numpy()[:, :W * H, 0]
print("seg ignores splats (geom ids only):", np.unique(seg).tolist())

# move group 0 up by 0.2 m: edit the splat arrays, then refit the splat BVH
p = rc.splat_position.numpy()
p[:N, 2] += 0.2
rc.splat_position.assign(p)
mjw.refit_splat_bvh(rc)
mjw.render(m, d, rc)
moved = wp.zeros((NWORLD, H, W), dtype=wp.vec3)
mjw.get_rgb(rc, 0, moved)
changed = np.any(np.abs(moved.numpy() - images["splats"]) > 1 / 255, axis=-1)
print("after moving group 0 + refit_splat_bvh, changed pixels per world:",
      changed.reshape(NWORLD, -1).sum(1).tolist())

if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    tiles = [images["no splats"][0], images["splats"][0], images["no splats"][1],
             images["splats"][1]]
    row = np.concatenate([np.pad(t, ((1, 1), (1, 1), (0, 0)), constant_values=1.0)
                          for t in tiles], axis=1)
    save_png(os.path.join(args.dump_png, "ex13_splats.png"),
             (np.clip(row, 0, 1) * 255).astype(np.uint8))
    print("wrote ex13_splats.png to", args.dump_png)

In [ ]:
# Example 27.13: world 0 / world 1 without and with their splat groups.
try:
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 4, figsize=(14, 3.6))
    tiles = [images["no splats"][0], images["splats"][0], images["no splats"][1], images["splats"][1]]
    for a, t, title in zip(axs, tiles, ["w0", "w0 + group 0", "w1", "w1 + group 1"]):
        a.imshow(np.clip(t, 0, 1), interpolation="nearest"); a.set_title(title); a.axis("off")
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## 14. A rendered rollout with CUDA graphs
On a GPU, warm up once, then capture a physics graph and a render graph and replay them per frame. On CPU the same calls run eagerly. The table prints your own device's numbers.

In [ ]:
# Example 27.11 - ex11_video_graph.py
"""Example 27.11 - a rendered rollout: physics graph + render graph (GPU), eager on CPU.

On a GPU we warm up once (JIT + megakernel build), then capture two CUDA graphs: one for
NSUB physics steps and one for refit_bvh + render. Each video frame replays both and unpacks
world 0. On CPU there are no CUDA graphs, so the same calls run eagerly with a tiny budget.
The frames/s table is for THIS device (CPU numbers on a CPU box are not GPU numbers).
Run: python ex11_video_graph.py [--dump-png DIR] [--gif PATH]
  --dump-png DIR  writes ex11_filmstrip.png (world 0, every frame side by side).
  --gif PATH      writes an animated GIF of world 0 (needs Pillow; skipped otherwise).
"""
import argparse
import os
import struct
import time
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

RENDER_SCENE = """
<mujoco model="render_scene">
  <option timestep="0.01666"/>
  <worldbody>
    <light name="top" pos="0 0 2" dir="0 0 -1" directional="true"/>
    <camera name="camera" pos="0 2 0" quat="0 0 1 1" resolution="128 128"/>
    <geom name="floor" type="plane" pos="0 0 -.5" size="2 2 .1" rgba=".9 .9 .9 1"/>
    <body name="box_and_sphere" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
      <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
      <geom name="green_sphere" pos="-.2 -.2 -.2" size=".1" rgba="0 1 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


parser = argparse.ArgumentParser()
parser.add_argument("--dump-png", default=None)
parser.add_argument("--gif", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLDS = [64, 1024, 8192] if ON_GPU else [1, 4]
W, H = (64, 64) if ON_GPU else (32, 32)
NFRAME = 60 if ON_GPU else 8
NSUB = 2                                        # physics steps per video frame

mjm = mujoco.MjModel.from_xml_string(RENDER_SCENE)


def rollout(nworld, keep_frames):
    m = mjw.put_model(mjm)
    if ON_GPU and not wp.is_conditional_graph_supported():
        m.opt.graph_conditional = False            # CUDA < 12.4: no conditional nodes
        print("note: conditional graphs unsupported, graph_conditional = False")
    d = mjw.make_data(mjm, nworld=nworld)
    qpos = np.tile(mjm.qpos0, (nworld, 1))
    qpos[:, 0] = np.linspace(0.3, 1.5, nworld)  # each world starts the swing at its own angle
    d.qpos.assign(qpos.astype(np.float32))
    rc = mjw.create_render_context(mjm, nworld=nworld, cam_res=(W, H), render_rgb=True,
                                   render_depth=False, render_seg=False)
    rgb = wp.zeros((nworld, H, W), dtype=wp.vec3)

    def physics():
        for _ in range(NSUB):
            mjw.step(m, d)

    def draw():
        mjw.refit_bvh(m, d, rc)
        mjw.render(m, d, rc)
        mjw.get_rgb(rc, 0, rgb)

    mjw.forward(m, d)
    physics()                                       # warm-up: JIT-compile step kernels
    draw()                                          # warm-up: build the megakernel (rc._megakernel)
    if ON_GPU:
        with wp.ScopedCapture() as cap_phys:
            physics()
        with wp.ScopedCapture() as cap_draw:
            draw()
        run_phys = lambda: wp.capture_launch(cap_phys.graph)
        run_draw = lambda: wp.capture_launch(cap_draw.graph)
    else:
        run_phys, run_draw = physics, draw          # CPU: eager (no CUDA graphs)

    frames = []
    wp.synchronize()
    t0 = time.perf_counter()
    for _ in range(NFRAME):
        run_phys()
        run_draw()
        if keep_frames:                             # reading back costs a sync + copy
            frames.append((np.clip(rgb.numpy()[0], 0, 1) * 255).astype(np.uint8))
    wp.synchronize()
    dt = time.perf_counter() - t0
    return frames, dt, d


frames, _, d = rollout(NWORLDS[0], keep_frames=True)
print(f"world 0: {len(frames)} frames of {frames[0].shape} uint8, "
      f"the loop simulates {NFRAME * NSUB * mjm.opt.timestep:.3f} s")
print("swing angle per world after the rollout:", np.round(d.qpos.numpy()[:4, 0], 3))
print("mean brightness of frames 0, 4, last:",
      [round(float(frames[i].mean()), 1) for i in (0, 4, len(frames) - 1)])
print(f"{'nworld':>7} {'frames/s (all worlds)':>22} {'images/s':>10}   [{wp.get_device()}]")
for nw in NWORLDS:
    _, dt, _ = rollout(nw, keep_frames=False)
    print(f"{nw:7d} {NFRAME / dt:22.1f} {NFRAME * nw / dt:10.0f}")

if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    strip = np.concatenate([np.pad(f, ((1, 1), (1, 1), (0, 0)), constant_values=255)
                            for f in frames], axis=1)
    save_png(os.path.join(args.dump_png, "ex11_filmstrip.png"), strip)
    print("wrote ex11_filmstrip.png", strip.shape, "to", args.dump_png)
if args.gif:
    try:
        from PIL import Image
        imgs = [Image.fromarray(f).resize((128, 128), Image.NEAREST) for f in frames]
        imgs[0].save(args.gif, save_all=True, append_images=imgs[1:], duration=66, loop=0)
        print("wrote", args.gif)
    except ImportError:
        print("skipped: Pillow not installed (no GIF)")

In [ ]:
# Play world 0 of Example 27.11 (mediapy is installed by the setup cell).
try:
    import mediapy as media
    media.show_video([np.repeat(np.repeat(f, 4, 0), 4, 1) for f in frames], fps=15)
except ImportError:
    print("skipped: mediapy not installed")

## Exercise 27.1: a pixel observation tensor
Fill in the kernel and `pixel_obs` so that the asserts pass: `(nworld, 3, H, W)` float32 in [0, 1], red channel = bits 16..23 of `rc.rgb_data`.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

RENDER_SCENE = """
<mujoco model="render_scene">
  <worldbody>
    <light name="top" pos="0 0 2" dir="0 0 -1" directional="true"/>
    <camera name="camera" pos="0 2 0" quat="0 0 1 1"/>
    <geom name="floor" type="plane" pos="0 0 -.5" size="2 2 .1" rgba=".9 .9 .9 1"/>
    <body name="box_and_sphere" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
      <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
      <geom name="green_sphere" pos="-.2 -.2 -.2" size=".1" rgba="0 1 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""
NWORLD, W, H = 4, 32, 24          # deliberately W != H


@wp.kernel
def rgb_to_nchw(rgb: wp.array3d(dtype=wp.vec3), obs: wp.array4d(dtype=float)):
    w, y, x = wp.tid()                      # launch over (nworld, H, W)
    # TODO: copy the three channels of rgb[w, y, x] into obs[w, 0..2, y, x]


def pixel_obs(m, d, rc, rgb, obs):
    # TODO: refit the BVH, render, unpack camera 0 into rgb, launch rgb_to_nchw
    return obs


mjm = mujoco.MjModel.from_xml_string(RENDER_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=(W, H), render_rgb=True,
                               render_depth=False, render_seg=False)
rgb = wp.zeros((NWORLD, H, W), dtype=wp.vec3)
obs = wp.zeros((NWORLD, 3, H, W), dtype=float)
q = np.tile(mjm.qpos0, (NWORLD, 1)); q[:, 0] = np.linspace(0, 3, NWORLD)
d.qpos.assign(q.astype(np.float32))
mjw.forward(m, d)
o = pixel_obs(m, d, rc, rgb, obs).numpy()

assert o.shape == (NWORLD, 3, H, W) and o.dtype == np.float32, (o.shape, o.dtype)
assert 0.0 <= o.min() and o.max() <= 1.0
packed = rc.rgb_data.numpy()[:, :W * H].reshape(NWORLD, H, W)
assert np.allclose(o[:, 0], ((packed >> 16) & 255) / 255.0)      # red channel = bits 16..23
assert not np.allclose(o[0], o[-1])                               # worlds differ
print("obs", o.shape, o.dtype, "range", o.min(), o.max(), "-> all checks passed")

In [ ]:
#@title Exercise 27.1: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

RENDER_SCENE = """
<mujoco model="render_scene">
  <worldbody>
    <light name="top" pos="0 0 2" dir="0 0 -1" directional="true"/>
    <camera name="camera" pos="0 2 0" quat="0 0 1 1"/>
    <geom name="floor" type="plane" pos="0 0 -.5" size="2 2 .1" rgba=".9 .9 .9 1"/>
    <body name="box_and_sphere" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
      <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
      <geom name="green_sphere" pos="-.2 -.2 -.2" size=".1" rgba="0 1 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""
NWORLD, W, H = 4, 32, 24          # deliberately W != H


@wp.kernel
def rgb_to_nchw(rgb: wp.array3d(dtype=wp.vec3), obs: wp.array4d(dtype=float)):
    w, y, x = wp.tid()                      # launch over (nworld, H, W)
    c = rgb[w, y, x]
    obs[w, 0, y, x] = c[0]
    obs[w, 1, y, x] = c[1]
    obs[w, 2, y, x] = c[2]


def pixel_obs(m, d, rc, rgb, obs):
    mjw.refit_bvh(m, d, rc)
    mjw.render(m, d, rc)
    mjw.get_rgb(rc, 0, rgb)
    wp.launch(rgb_to_nchw, dim=rgb.shape, inputs=[rgb], outputs=[obs])
    return obs


mjm = mujoco.MjModel.from_xml_string(RENDER_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=(W, H), render_rgb=True,
                               render_depth=False, render_seg=False)
rgb = wp.zeros((NWORLD, H, W), dtype=wp.vec3)
obs = wp.zeros((NWORLD, 3, H, W), dtype=float)
q = np.tile(mjm.qpos0, (NWORLD, 1)); q[:, 0] = np.linspace(0, 3, NWORLD)
d.qpos.assign(q.astype(np.float32))
mjw.forward(m, d)
o = pixel_obs(m, d, rc, rgb, obs).numpy()

assert o.shape == (NWORLD, 3, H, W) and o.dtype == np.float32, (o.shape, o.dtype)
assert 0.0 <= o.min() and o.max() <= 1.0
packed = rc.rgb_data.numpy()[:, :W * H].reshape(NWORLD, H, W)
assert np.allclose(o[:, 0], ((packed >> 16) & 255) / 255.0)      # red channel = bits 16..23
assert not np.allclose(o[0], o[-1])                               # worlds differ
print("obs", o.shape, o.dtype, "range", o.min(), o.max(), "-> all checks passed")

## Exercise 27.2: lidar minimum range against geometry
Implement `min_range` so that it matches the exact distance to the nearest wall or pillar within 1 mm, with and without the BVH.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec6

ROOM = """
<mujoco><worldbody>
  <geom name="floor" type="plane" size="3 3 .1"/>
  <geom type="box" pos=" 2 0 .5" size=".05 2 .5"/><geom type="box" pos="-2 0 .5" size=".05 2 .5"/>
  <geom type="box" pos="0  2 .5" size="2 .05 .5"/><geom type="box" pos="0 -2 .5" size="2 .05 .5"/>
  <geom name="pillar" type="cylinder" pos="1 1 .4" size=".2 .4"/>
  <body name="robot" pos="0 0 .15"><freejoint/><geom type="cylinder" size=".15 .15"/></body>
</worldbody></mujoco>"""
XY = np.array([[0.0, 0.0], [1.5, -0.4], [0.6, 0.7], [-1.2, 1.6]])
NWORLD, NRAY = len(XY), 360


def min_range(m, d, robot, nray=NRAY, rc=None):
    """Smallest lidar range per world, rays in the horizontal plane at the robot origin."""
    # TODO: build pnt / vec of shape (nworld, nray), outputs of shape (nworld, nray),
    #       call mjw.rays with bodyexclude = robot for every ray, ignore misses (-1), reduce
    raise NotImplementedError


def analytic(xy):
    wall = 1.95 - np.abs(xy).max(1)
    pillar = np.linalg.norm(xy - [1.0, 1.0], axis=1) - 0.2
    return np.minimum(wall, pillar)


mjm = mujoco.MjModel.from_xml_string(ROOM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
q = np.tile(mjm.qpos0, (NWORLD, 1)); q[:, :2] = XY
d.qpos.assign(q.astype(np.float32))
mjw.kinematics(m, d)
robot = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "robot")
got, want = min_range(m, d, robot), analytic(XY)
print("lidar :", got.round(4))
print("exact :", want.round(4))
assert np.allclose(got, want, atol=1e-3), (got, want)
rc = mjw.create_render_context(mjm, nworld=NWORLD)
mjw.refit_bvh(m, d, rc)
assert np.allclose(min_range(m, d, robot, rc=rc), got, atol=1e-6)
print("min_range matches the geometry in all", NWORLD, "worlds (with and without the BVH)")

In [ ]:
#@title Exercise 27.2: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec6

ROOM = """
<mujoco><worldbody>
  <geom name="floor" type="plane" size="3 3 .1"/>
  <geom type="box" pos=" 2 0 .5" size=".05 2 .5"/><geom type="box" pos="-2 0 .5" size=".05 2 .5"/>
  <geom type="box" pos="0  2 .5" size="2 .05 .5"/><geom type="box" pos="0 -2 .5" size="2 .05 .5"/>
  <geom name="pillar" type="cylinder" pos="1 1 .4" size=".2 .4"/>
  <body name="robot" pos="0 0 .15"><freejoint/><geom type="cylinder" size=".15 .15"/></body>
</worldbody></mujoco>"""
XY = np.array([[0.0, 0.0], [1.5, -0.4], [0.6, 0.7], [-1.2, 1.6]])
NWORLD, NRAY = len(XY), 360


def min_range(m, d, robot, nray=NRAY, rc=None):
    """Smallest lidar range per world, rays in the horizontal plane at the robot origin."""
    ang = 2 * np.pi * np.arange(nray) / nray
    origin = d.xpos.numpy()[:, robot]
    pnt = wp.array(np.repeat(origin[:, None], nray, 1), dtype=wp.vec3)
    vec = wp.array(np.tile(np.stack([np.cos(ang), np.sin(ang), 0 * ang], 1), (d.nworld, 1, 1)),
                   dtype=wp.vec3)
    dist = wp.zeros((d.nworld, nray), dtype=float)
    gid = wp.zeros((d.nworld, nray), dtype=int)
    nrm = wp.zeros((d.nworld, nray), dtype=wp.vec3)
    mjw.rays(m, d, pnt, vec, vec6(-1, -1, -1, -1, -1, -1), True,
             wp.full(nray, robot, dtype=int), dist, gid, nrm, rc=rc)
    r = dist.numpy()
    return np.where(r < 0, np.inf, r).min(1)


def analytic(xy):
    wall = 1.95 - np.abs(xy).max(1)
    pillar = np.linalg.norm(xy - [1.0, 1.0], axis=1) - 0.2
    return np.minimum(wall, pillar)


mjm = mujoco.MjModel.from_xml_string(ROOM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
q = np.tile(mjm.qpos0, (NWORLD, 1)); q[:, :2] = XY
d.qpos.assign(q.astype(np.float32))
mjw.kinematics(m, d)
robot = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "robot")
got, want = min_range(m, d, robot), analytic(XY)
print("lidar :", got.round(4))
print("exact :", want.round(4))
assert np.allclose(got, want, atol=1e-3), (got, want)
rc = mjw.create_render_context(mjm, nworld=NWORLD)
mjw.refit_bvh(m, d, rc)
assert np.allclose(min_range(m, d, robot, rc=rc), got, atol=1e-6)
print("min_range matches the geometry in all", NWORLD, "worlds (with and without the BVH)")

## Stretch: an orbiting camera
In every frame: `step`, then write `d.cam_xpos`/`d.cam_xmat`, then `refit_bvh` and `render`. Then move the write before `step` and explain what you see. On a GPU, raise the resolution and capture the render in a graph.

In [ ]:
# your orbit here

In [ ]:
#@title Stretch: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

RENDER_SCENE = """
<mujoco model="render_scene">
  <option timestep="0.01666"/>
  <worldbody>
    <light name="top" pos="0 0 2" dir="0 0 -1" directional="true"/>
    <camera name="camera" pos="0 2 0" quat="0 0 1 1"/>
    <geom name="floor" type="plane" pos="0 0 -.5" size="2 2 .1" rgba=".9 .9 .9 1"/>
    <body name="box_and_sphere" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
      <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
      <geom name="green_sphere" pos="-.2 -.2 -.2" size=".1" rgba="0 1 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""


def look_at(eye, target, up=(0.0, 0.0, 1.0)):
    """cam_xmat columns = camera x, y, z axes in world; the camera looks along -z."""
    fwd = np.asarray(target, float) - eye
    fwd /= np.linalg.norm(fwd)
    x = np.cross(fwd, up); x /= np.linalg.norm(x)
    y = np.cross(-fwd, x)
    return np.column_stack([x, y, -fwd])


NWORLD, W, H, NFRAME = 2, 32, 32, 12
mjm = mujoco.MjModel.from_xml_string(RENDER_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=(W, H), render_rgb=True,
                               render_depth=False, render_seg=False)
rgb = wp.zeros((NWORLD, H, W), dtype=wp.vec3)
frames = []
mjw.forward(m, d)
for f in range(NFRAME):
    mjw.step(m, d)                                     # 1. physics (camlight runs inside)
    a = 2 * np.pi * f / NFRAME                         # 2. THEN overwrite the camera
    eye = np.array([2 * np.sin(a), 2 * np.cos(a), 0.6])
    pos, mat = d.cam_xpos.numpy(), d.cam_xmat.numpy()
    pos[:, 0], mat[:, 0] = eye, look_at(eye, [0, 0, -0.2])
    d.cam_xpos.assign(pos)
    d.cam_xmat.assign(mat)
    mjw.refit_bvh(m, d, rc)                            # 3. refit + render read the new camera
    mjw.render(m, d, rc)
    mjw.get_rgb(rc, 0, rgb)
    frames.append((np.clip(rgb.numpy()[0], 0, 1) * 255).astype(np.uint8))
diffs = [np.abs(frames[i + 1].astype(int) - frames[i]).mean() for i in range(NFRAME - 1)]
print(len(frames), "orbit frames; mean change between frames:", np.round(diffs, 1))

In [ ]:
# Show the orbit as a video.
try:
    import mediapy as media
    media.show_video([np.repeat(np.repeat(f, 4, 0), 4, 1) for f in frames], fps=6)
except ImportError:
    print("skipped: mediapy not installed")